# Package A: Joining the waste and population data

**Team 2: Smart Waste Analytics & Demographics** (owner: Birkás Levente)

This notebook joins the cleaned waste tables from package B with the cleaned population data from package C,
so that D, E and F can work from one table.

**Inputs** (all in `processed/`)
- from B: `municipal_waste_collected_clean.csv`, `municipal_waste_recycled_pct_clean.csv`, `municipal_waste_treatment_latest_clean.csv`
- from C: `population_clean.csv`, `country_region_lookup.csv`

**Outputs** (saved to `processed/`)
- `panel_country_year.csv`: one row per country and year (2001–2024), waste + recycling + population + region + waste per person
- `treatment_latest_joined.csv`: one row per country, latest-year treatment shares + region + population

**Steps**
1. Load the five input files
2. Join waste collected and recycling percent on country code and year
3. Add population, growth rate and region
4. Calculate waste per person
5. Reshape the treatment snapshot to one row per country and check its quality
6. Final checks and save

The join key is the UN M49 country code (`CountryID`) plus `Year`. Country names are never used for matching.

## 1. Setup and file paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Works whether the notebook is run from team2_waste_data/scripts/,
# from team2_waste_data/, or from the repository root (same logic as package C).
HERE = Path.cwd()
if HERE.name == "scripts":
    DATA_DIR = HERE.parent
elif (HERE / "team2_waste_data").exists():
    DATA_DIR = HERE / "team2_waste_data"
else:
    DATA_DIR = HERE

PROCESSED = DATA_DIR / "processed"

# Countries with waste data but no population data in WPP 2024 (found by package C).
#   736 Sudan (Former): Sudan before South Sudan split off in 2011; WPP only has today's Sudan (729) and South Sudan (728)
#   383 Kosovo: WPP 2024 has no separate row for Kosovo
EXPECTED_UNMATCHED = {736, 383}

print("Reading from and writing to:", PROCESSED.resolve())

Reading from and writing to: C:\Users\Levente\OneDrive - Széchenyi István Egyetem\Egyetem\Data Analysis\team-smart-waste\team2_waste_data\processed


## 2. Load the input files

In [2]:
def load(name):
    """Read one CSV from processed/ and stop with a clear message if it is missing."""
    path = PROCESSED / name
    if not path.exists():
        raise FileNotFoundError(f"{name} not found. Run the cleaning notebooks of packages B and C first.")
    return pd.read_csv(path)


collected = load("municipal_waste_collected_clean.csv")        # B: waste collected, 1000 tonnes, 2001-2023
recycled = load("municipal_waste_recycled_pct_clean.csv")      # B: % of collected waste recycled, 2001-2024
treatment = load("municipal_waste_treatment_latest_clean.csv")  # B: treatment shares, latest year per country
population = load("population_clean.csv")                       # C: population 1950-2100
regions = load("country_region_lookup.csv")                     # C: country -> sub-region -> region

for name, df in [("collected", collected), ("recycled", recycled), ("treatment", treatment),
                 ("population", population), ("regions", regions)]:
    print(f"{name:<11} {str(df.shape):<14} {df['CountryID'].nunique():>4} countries")

collected   (3289, 5)       143 countries
recycled    (2520, 5)       105 countries
treatment   (636, 7)        106 countries
population  (35787, 8)      237 countries
regions     (237, 5)        237 countries


## 3. Join waste collected and recycling percent

Both tables have one row per country and year. We use an **outer join**: 143 countries report waste collected and
105 report a recycling percent, but only 104 report both. An outer join keeps the countries that are in only one table.

Each file has its own `Source` column (U = UNSD questionnaire, E = Eurostat, O = OECD), and for a few
country-years they differ, so both are kept.

Rows where both values are missing are dropped: they carry no information.

In [3]:
waste = (
    collected.rename(columns={"Source": "Source_Collected"})
    .drop(columns="Country")
    .merge(recycled.rename(columns={"Source": "Source_Recycled"}).drop(columns="Country"),
           on=["CountryID", "Year"], how="outer")
)

has_value = waste[["Waste_Collected_1000t", "Recycled_Pct"]].notna().any(axis=1)
print("Country-year rows after the outer join:", len(waste))
print("Rows with at least one waste value:    ", has_value.sum())
waste = waste[has_value].copy()

assert not waste.duplicated(["CountryID", "Year"]).any(), "Duplicate country-year rows in the waste join"

Country-year rows after the outer join: 3417
Rows with at least one waste value:     2542


## 4. Add population, growth rate and region

A **left join** keeps every waste row and adds the population of the same country in the same year,
then the region of the country. Afterwards we list every waste country that found no population match.

In [4]:
panel = (
    waste
    .merge(population.drop(columns=["ISO3", "Country", "Is_Projection"]), on=["CountryID", "Year"], how="left")
    .merge(regions, on="CountryID", how="left")
)

no_match = panel[panel["Population_1000"].isna()]
unmatched_ids = set(no_match["CountryID"])
names = pd.concat([collected, recycled]).drop_duplicates("CountryID").set_index("CountryID")["Country"]

print("Waste rows without a population match:", len(no_match))
for cid in sorted(unmatched_ids):
    print(f"  {cid}: {names[cid]} ({(no_match['CountryID'] == cid).sum()} rows)")

# Anything beyond the two known cases means the cleaning steps changed and must be looked at.
assert unmatched_ids <= EXPECTED_UNMATCHED, f"Unexpected unmatched countries: {unmatched_ids - EXPECTED_UNMATCHED}"

panel = panel[panel["Population_1000"].notna()].copy()
assert panel["Region"].notna().all(), "Some matched countries have no region"
print("Rows kept:", len(panel), "| countries:", panel["CountryID"].nunique())

Waste rows without a population match: 10
  383: Kosovo* (2 rows)
  736: Sudan (Former) (8 rows)
Rows kept: 2532 | countries: 142


The two dropped countries are expected: neither has population data in WPP 2024 (see package C's notebook).
Without population we cannot compute waste per person, so they are left out of the panel.

## 5. Waste per person

`Waste_Collected_1000t` is in thousand tonnes and `Population_1000` in thousand people, so

kg per person = (1000 t × 1000 kg/t) / (1000 people) = `Waste_Collected_1000t / Population_1000 × 1000`

In [5]:
panel["Waste_kg_per_capita"] = panel["Waste_Collected_1000t"] / panel["Population_1000"] * 1000

print(panel["Waste_kg_per_capita"].describe().round(1).to_string())

# Flag unusual values for package D to look at. Nothing is removed here.
per_country = panel.groupby("Country")["Waste_kg_per_capita"].median().dropna()
print("\nMedian kg per person below 50:", per_country[per_country < 50].round(0).to_dict())
print("Median kg per person above 1500:", per_country[per_country > 1500].round(0).to_dict())
print("\nHungary, last 3 years:")
print(panel.loc[panel["CountryID"] == 348, ["Year", "Waste_kg_per_capita"]].dropna().tail(3).round(1).to_string(index=False))

count    2056.0
mean      498.5
std       463.9
min         0.0
25%       285.1
50%       434.4
75%       593.3
max      7514.6

Median kg per person below 50: {'Bangladesh': 31.0, 'Benin': 25.0, 'Burkina Faso': 44.0, 'Burundi': 2.0, 'Cambodia': 27.0, 'India': 40.0, 'Indonesia': 27.0, 'Kenya': 23.0, 'Madagascar': 19.0, 'Myanmar': 7.0, 'Nepal': 23.0, 'Samoa': 44.0, 'Senegal': 41.0, 'Togo': 26.0, 'United Republic of Tanzania': 7.0, 'Viet Nam': 0.0, 'Zambia': 33.0}
Median kg per person above 1500: {'Democratic Republic of the Congo': 6329.0, 'Kuwait': 3414.0}

Hungary, last 3 years:
 Year  Waste_kg_per_capita
 2021                416.4
 2022                403.8
 2023                424.8


Most countries are between about 300 and 600 kg per person per year, which matches typical published figures
(Hungary ≈ 400–425). The values flagged above are **not removed**: very low values often mean that only part of the
population is served by collection, and very high ones are small islands, tourism or likely reporting errors
(for example DR Congo). Package D decides how to treat them and explains it.

## 6. Treatment snapshot: one row per country

B's treatment table is in long format (one row per country and treatment type). Here it is turned into one row per
country, with one column per treatment type.

**Data quality problem in the raw UNSD file.** The shares do not always add up to 100%:
- **Eurostat countries (Source E)**: "incinerated" only covers incineration *without* energy recovery (code D10),
  according to the Description sheet of the Excel file. Incineration with energy recovery is not in the table at all,
  so these countries add up to only about 46–100%. Their shares are kept but marked `incomplete`.
- **OECD countries (Source O)**, e.g. USA, Japan, Korea, UK, Australia: the shares add up to 0–3%, which is impossible.
  These are a unit problem in the source file and cannot be repaired reliably, so their shares are set to missing and
  marked `invalid`. Their total amount managed (`Managed_1000t`) is kept.
- **UNSD questionnaire countries (Source U)**: mostly add up to 100% and are marked `complete`. A few are clearly
  wrong (for example Algeria 0%, Malaysia 334%) and are marked `invalid` by the same rule.

Rule used: sum of shares below 5% or above 105% → `invalid`, 5–95% → `incomplete`, 95–105% → `complete`.

In [6]:
TREATMENT_COLUMNS = {
    "Municipal waste managed": "Managed_1000t",
    "Municipal waste landfilled": "Landfilled_Pct",
    "Municipal waste incinerated": "Incinerated_Pct",
    "Municipal waste recycled": "Recycled_Pct",
    "Municipal waste composted": "Composted_Pct",
    "Municipal waste managed with other treatment or disposal": "Other_Pct",
}
SHARE_COLUMNS = [c for c in TREATMENT_COLUMNS.values() if c.endswith("_Pct")]

wide = (
    treatment.pivot(index="CountryID", columns="Treatment_Type", values="Value")
    .rename(columns=TREATMENT_COLUMNS)
    .reset_index()
)
wide.columns.name = None
info = treatment.drop_duplicates("CountryID")[["CountryID", "Source", "Year"]]
wide = info.merge(wide, on="CountryID")


def share_quality(row):
    """Label how trustworthy a country's treatment shares are, based on how close their sum is to 100%."""
    total = row[SHARE_COLUMNS].sum(min_count=1)
    if pd.isna(total):
        return "no data"
    if total < 5 or total > 105:
        return "invalid"
    if total < 95:
        return "incomplete"
    return "complete"


wide["Shares_Sum_Pct"] = wide[SHARE_COLUMNS].sum(axis=1, min_count=1)
wide["Shares_Quality"] = wide.apply(share_quality, axis=1)
print(pd.crosstab(wide["Source"], wide["Shares_Quality"]))
print("\nInvalid rows (shares set to missing):")
print(wide.loc[wide["Shares_Quality"] == "invalid", ["CountryID", "Source", "Shares_Sum_Pct"]]
      .merge(regions[["CountryID", "Country"]], on="CountryID", how="left").round(2).to_string(index=False))

wide.loc[wide["Shares_Quality"] == "invalid", SHARE_COLUMNS] = np.nan
assert not wide["CountryID"].duplicated().any(), "Duplicate countries in the treatment table"

Shares_Quality  complete  incomplete  invalid  no data
Source                                                
E                      4          32        0        0
O                      0           0       10        0
U                     44           5        5        6

Invalid rows (shares set to missing):
 CountryID Source  Shares_Sum_Pct                  Country
        12      U            0.00                  Algeria
        36      O            0.68                Australia
       152      O            1.00                    Chile
       446      U          120.27         China, Macao SAR
       188      O            1.50               Costa Rica
       340      U            2.30                 Honduras
       392      O            0.25                    Japan
       458      U          334.38                 Malaysia
       484      O            0.14                   Mexico
       554      O            2.67              New Zealand
       410      O            0.43    

Now add the region and the population **of the year each country reported** (it differs per country, 2002–2024),
and the amount managed per person.

In [7]:
treatment_joined = (
    wide
    .merge(regions, on="CountryID", how="left")
    .merge(population[["CountryID", "Year", "Population_1000"]], on=["CountryID", "Year"], how="left")
)

t_unmatched = set(treatment_joined.loc[treatment_joined["Population_1000"].isna(), "CountryID"])
print("Treatment countries without a population match:", sorted(t_unmatched))
assert t_unmatched <= EXPECTED_UNMATCHED, f"Unexpected unmatched countries: {t_unmatched - EXPECTED_UNMATCHED}"
treatment_joined = treatment_joined[treatment_joined["Population_1000"].notna()].copy()

treatment_joined["Managed_kg_per_capita"] = treatment_joined["Managed_1000t"] / treatment_joined["Population_1000"] * 1000
print("Countries kept:", len(treatment_joined))
print(treatment_joined["Shares_Quality"].value_counts().to_string())

Treatment countries without a population match: [383]
Countries kept: 105
Shares_Quality
complete      47
incomplete    37
invalid       15
no data        6


## 7. Final checks and save

In [8]:
PANEL_COLUMNS = [
    "CountryID", "ISO3", "Country", "SubRegion", "Region", "Year",
    "Source_Collected", "Source_Recycled", "Waste_Collected_1000t", "Recycled_Pct",
    "Population_1000", "Growth_Rate_Pct", "Pop_Density_per_km2", "Waste_kg_per_capita",
]
TREATMENT_OUT_COLUMNS = [
    "CountryID", "ISO3", "Country", "SubRegion", "Region", "Year", "Source",
    "Managed_1000t", *SHARE_COLUMNS, "Shares_Sum_Pct", "Shares_Quality",
    "Population_1000", "Managed_kg_per_capita",
]

panel = panel[PANEL_COLUMNS].sort_values(["CountryID", "Year"]).reset_index(drop=True)
treatment_joined = treatment_joined[TREATMENT_OUT_COLUMNS].sort_values("CountryID").reset_index(drop=True)
panel["Year"] = panel["Year"].astype(int)
treatment_joined["Year"] = treatment_joined["Year"].astype(int)

assert not panel.duplicated(["CountryID", "Year"]).any()
assert panel[["ISO3", "Country", "Region", "Population_1000"]].notna().all().all()
assert (panel["Waste_kg_per_capita"].dropna() >= 0).all()
assert treatment_joined[["Country", "Region"]].notna().all().all()

print(f"Panel: {len(panel)} rows, {panel['CountryID'].nunique()} countries, years {panel['Year'].min()}-{panel['Year'].max()}")
print(f"Treatment: {len(treatment_joined)} countries")
print("\nCountries per region in the panel:")
print(panel.drop_duplicates("CountryID")["Region"].value_counts().to_string())
print("\nShare of missing values in the panel (%):")
print((panel.isna().mean() * 100).round(1)[lambda s: s > 0].to_string())

panel.to_csv(PROCESSED / "panel_country_year.csv", index=False)
treatment_joined.to_csv(PROCESSED / "treatment_latest_joined.csv", index=False)
print("\nSaved panel_country_year.csv and treatment_latest_joined.csv")

Panel: 2532 rows, 142 countries, years 2001-2024
Treatment: 105 countries

Countries per region in the panel:
Region
Europe      42
Asia        37
Americas    33
Africa      24
Oceania      6

Share of missing values in the panel (%):
Source_Collected          1.7
Source_Recycled          16.0
Waste_Collected_1000t    18.8
Recycled_Pct             41.3
Waste_kg_per_capita      18.8

Saved panel_country_year.csv and treatment_latest_joined.csv


## Output summary

| File | One row per | Main columns |
|---|---|---|
| `panel_country_year.csv` | country and year (2001–2024) | CountryID, ISO3, Country, SubRegion, Region, Year, Waste_Collected_1000t, Recycled_Pct, Population_1000, Growth_Rate_Pct, Waste_kg_per_capita |
| `treatment_latest_joined.csv` | country (latest year it reported) | CountryID, ISO3, Country, Region, Year, Source, Managed_1000t, Landfilled_Pct, Incinerated_Pct, Recycled_Pct, Composted_Pct, Other_Pct, Shares_Quality, Managed_kg_per_capita |

**Notes for D, E and F**
- Use `ISO3` for Plotly maps and `Region` / `SubRegion` for regional comparisons.
- Many country-years have no waste value (see the missing-value summary above), so check coverage before comparing years.
- For population projections to 2100, use C's `population_clean.csv` directly (it has `Is_Projection`).
- In the treatment table, only use rows with `Shares_Quality == "complete"` when you need shares that add up to 100%.
  `incomplete` rows (Eurostat) are missing incineration with energy recovery; `invalid` rows have no shares.